# Main experiment matrix — batch 2

Same protocol as `02_main_exps_1.ipynb` (45k train / 5k val, seeds **0 / 42 / 123**, all decisions on val,
collapsed runs kept and flagged). Runs are saved to the same `runs/` directory, so they show up
automatically in **`03_final_analysis.ipynb`** — the master table is no longer generated here.

**编号规则：用 `exp_2xx`**（第二批），不要复用 `02_main_exps_1.ipynb` 里已占用的 `exp_101`–`exp_109`。
同一个 exp 号 = 同一个配置，只是种子不同 → 主表按 `exp` 分组算 mean ± std。

**依赖约定：**先运行 code #1（导入）和 code #2（共享 loaders），之后可以直接运行任意一个实验 cell；
code #3 只是 S1 的配置预览，不是 code #4–#11 的前置依赖。

Reminder before long runs: 确认笔记本性能模式 + 跑下面那个 15 秒 GPU 自检。

In [1]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.config import PROJECT_ROOT, DATA_DIR, RUNS_DIR, SEED, CIFAR100_MEAN, CIFAR100_STD
from src.models import baseline_cnn, baseline_cnn_bn, cnn_3x3, cnn_down
from src.models import (cnn_residual, cnn_wide, cnn_deep, cnn_3stage,
                        cnn_dw, cnn_dw_wide, cnn_groupnorm, cnn_deep_residual,
                        cnn_deep_residual_wide,
                        cnn_deep_residual_d3, cnn_deep_residual_d4, cnn_wide3)   # batch-2 架构候选
from src.data import build_datasets, build_loaders, build_loaders_val
from src.engine import train_one_epoch, evaluate, fit
from src.utils import set_seed, save_run, check_gpu, warmup_cosine
from src.experiments import (
    run_experiment,
    run_batch,
    build_experiment_metadata,
    build_architecture_ablation_batch,
)

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

cuda NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# Shared loaders: no-aug (NOAUG) and augmented (AUG), with the same 45k/5k split.
#
# num_workers：2026-10-16 实测（cnn_down + crop/flip，128 batch）—— nw=0 时 58.9 ms/batch（20.7 s/epoch），
# nw=2 或 4 都是 22.8 ms/batch（8.0 s/epoch），**正好贴住纯 GPU 下限 22.6 ms**。即：瓶颈是单进程取数，不是 GPU。
# 取 2 而不是 4：两者同速，但每个 worker 都会独立 import torch 并持有一份数据集副本，少开进程更省内存。
# 第 1 个 batch 会有一次性的 ~7-13 s worker 启动开销（persistent_workers 保证每个 loader 只付一次）。
#
# ⚠️ 若在 Jupyter 里卡死在第一个 batch（Windows spawn 的老问题），把 NUM_WORKERS 改回 0 即可，其余一切不变。
NUM_WORKERS = 2

train_loader, val_loader, test_loader = build_loaders_val(batch_size=128, num_workers=NUM_WORKERS)

from torchvision import transforms
aug_train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])
aug_loader, aug_val, _ = build_loaders_val(batch_size=128, num_workers=NUM_WORKERS, train_tf=aug_train_tf)

# Cutout（RandomErasing）：在 crop+flip 之上再随机遮掉一小块（p=0.5）。
# 单独一组 loader —— exp_220 相对 exp_212 只多这一个变量。
erase_train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0),
])
erase_loader, erase_val, _ = build_loaders_val(batch_size=128, num_workers=NUM_WORKERS, train_tf=erase_train_tf)

# 下面这个 cell 负责构造三个 loader 入口，真正的训练都交给后面的 cell 里去一步步完成。
NOAUG = (train_loader, val_loader)
AUG = (aug_loader, aug_val)
ERASE = (erase_loader, erase_val)

print(f"no-aug: train {len(train_loader)} batches | val {len(val_loader)} batches")
print(f"aug:    train {len(aug_loader)} batches | val {len(aug_val)} batches")
print(f"erase:  train {len(erase_loader)} batches | val {len(erase_val)} batches")

# ⚠️ 可比性（要写进报告）：num_workers>0 时增强的随机裁剪来自 worker 自己的 RNG，所以带增强的新 run 与
# batch-1（nw=0）的 aug run **不是逐位可比的**（统计上无偏）。无增强的实验完全不受影响
# （变换是确定性的，shuffle 顺序仍由主进程的全局 RNG 决定，seed 协议不变）。

no-aug: train 351 batches | val 40 batches
aug:    train 351 batches | val 40 batches
erase:  train 351 batches | val 40 batches


**跑批基础设施都在 `src/` 里，和第一批共用：** `src/experiments.py:run_experiment`（多 seed 跑批 + 组图 + 坍缩/慢态警告）。
`src/experiments.py:build_experiment_metadata`（不训练，只写 `config.json` 的元数据字段）。
`src/utils.py:check_gpu`（GPU 自检）、`src/utils.py:warmup_cosine`（lr 调度）、`src/engine.py:fit`（单次训练）。
每个 cell 都只依赖上面已经运行过一遍的 code #1 和 code #2 的导入。

---

# S1 · 架构消融（20 epochs · aug ON · 单变量）

**为什么是 20ep + 带增强**：exp_109（cnn_down 无增强）在 **10 个 epoch 内把训练集背到 96%**（val 只有 59%），
这个架构早就进了过拟合区。在无增强下做架构筛选会**系统性惩罚容量更大的模型**（它们只会背得更狠），
所以筛选必须待在**和最终长训同一个正则化区间**里。代价只是 epoch 时间从 18s 涨到 ~25s。

**尺子（anchor）**：`exp_200` = 底座 `cnn_down` @20ep、**3 seeds** —— 201–207 每一行的 Δ 都对着它算。
其余每个 = 底座 + **恰好一个**改动、**1 seed**（两级预算的筛选档，任务书规则二），胜者进 S1b 用 3 seeds 复核。

| exp | 改动 | 因素 | 参数量 |
|---|---|---|---|
| 200 | 底座（锚点，3 seeds） | — | 2.35M |
| 201 | + 残差连接（形状不变） | 7b | 2.46M |
| 202 | 宽度 ×2 | 7f | 9.35M |
| 203 | 深度 ×2（每 stage 2→4 个 3×3） | 7f | 5.09M |
| 204 | 去掉 4px 阶段（4→3 stage） | 7d | 1.17M |
| 205 | depthwise 可分离（形状不变） | 7c | 0.30M |
| 206 | depthwise + 宽度回投（≈同参数量） | 7c | 2.45M |
| 207 | BatchNorm → GroupNorm(8) | 7e | 2.35M |

> ⚠️ **本区块的数字只能在区块内比**（20ep / aug ON）。和第一批的 10ep 数字**不可混表** —— 任务书反面案例点名了这条。
> 跨预算的比较只出现在"epoch 阶梯"那张图上（exp_215），不在这里。


S1 的批次由 `build_architecture_ablation_batch()` 统一构造：code #3 只做预览，code #4 才真正启动训练，两格之间没有任何隐式的依赖关系。

In [ ]:
# S1 批次定义 现在 已经都交给 src/experiments.py 统一构造。
# 本 cell 只负责打印配置做预览，而训练在后面的 cell 里面去执行。
S1_EPOCHS = 20
S1_BATCH = build_architecture_ablation_batch()

print(f"{len(S1_BATCH)} 项：1 个 3-seed 锚点 + 7 个 1-seed 候选 | epochs={S1_EPOCHS}")
for c in S1_BATCH:
    n = sum(p.numel() for p in c["model"]().parameters())
    print(f"  {c['exp']}_{c['name']:<10} {n:>10,} params   seeds={c['seeds']}")

In [4]:
# 跑 S1（约 1.5–2 h，机器别睡）。
# 本 cell 依赖 code #1 + code #2 之后如果中途断了就直接重跑即可。
# skip_existing 会跳过已经落盘完成的配置，不会重做。
S1_EPOCHS = 20
s1_summary = run_batch(build_architecture_ablation_batch(), loaders=AUG, epochs=S1_EPOCHS)

epoch  1/20 | train 4.003/0.108 | test 3.616/0.153 | 21s
epoch  2/20 | train 3.382/0.184 | test 3.004/0.245 | 21s
epoch  3/20 | train 2.813/0.282 | test 2.866/0.291 | 21s
epoch  4/20 | train 2.354/0.379 | test 2.311/0.399 | 22s
epoch  5/20 | train 2.040/0.444 | test 2.117/0.441 | 22s
epoch  6/20 | train 1.817/0.495 | test 1.856/0.497 | 22s
epoch  7/20 | train 1.637/0.541 | test 1.817/0.501 | 22s
epoch  8/20 | train 1.485/0.580 | test 1.711/0.541 | 25s
epoch  9/20 | train 1.359/0.611 | test 1.619/0.573 | 24s
epoch 10/20 | train 1.237/0.641 | test 1.624/0.567 | 25s
epoch 11/20 | train 1.138/0.669 | test 1.457/0.608 | 25s
epoch 12/20 | train 1.037/0.696 | test 1.403/0.627 | 24s
epoch 13/20 | train 0.946/0.721 | test 1.315/0.641 | 24s
epoch 14/20 | train 0.867/0.743 | test 1.322/0.642 | 24s
epoch 15/20 | train 0.787/0.765 | test 1.268/0.658 | 24s
epoch 16/20 | train 0.730/0.786 | test 1.228/0.664 | 24s
epoch 17/20 | train 0.673/0.803 | test 1.210/0.677 | 24s
epoch 18/20 | train 0.627/0.816

---

## S1b · 冠军复核（3 seeds）+ exp_208（深度 + 残差）

- **exp_202 复核**：宽度 ×2 在 S1 里是 **1 seed** 打出的 +3.79，必须用 3 seeds 复核才能写进报告。
  锚点 exp_200 本身就是 3 seeds，所以 Δ 的尺子已经干净。
  ⚠️ 复核用**新的 run 名 `wide_conf`**（不是覆盖原来那个 1-seed 的 `wide`）—— 两个都留在 `runs/` 里，
  主表上会看到 `tier=screen` 和 `tier=confirm` 两行，这正是"两级预算"协议的可追溯记录。
- **exp_208 = 深度 ×2 + 残差**：我们手上已有两个干净的负结果 —— 残差单独（201，−0.02）、深度单独（203，−3.11）。
  这一项验证经典假设 **"残差连接解锁深度"**：两者组合起来能不能翻正？
  这是一个完整的"假设 → 实验 → 结论 → 下一步假设"闭环（报告里正是要这个）。

预计 ~40 min（@ NUM_WORKERS=2）。


code #5 只要 code #1 + code #2 之后，就永远都不会用到 S1 训练出来的结果。

In [3]:
# S1b + exp_208（约 40 min）。本 cell 只依赖 code #1 + code #2。
# 跑法相同：断了就直接重跑本格即可。
S1B_EPOCHS = 20
S1B_BATCH = [
    {"exp": "exp_202", "name": "wide_conf", "model": cnn_wide, "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+wide", "confirm")},
    {"exp": "exp_208", "name": "deep_residual", "model": cnn_deep_residual, "seeds": (0,),
     "extra": build_experiment_metadata("down+deep+residual", "screen")},
]
run_batch(S1B_BATCH, loaders=AUG, epochs=S1B_EPOCHS)

epoch  1/20 | train 3.749/0.139 | test 3.253/0.215 | 36s
epoch  2/20 | train 3.124/0.229 | test 2.887/0.271 | 25s
epoch  3/20 | train 2.621/0.317 | test 2.708/0.324 | 25s
epoch  4/20 | train 2.167/0.416 | test 2.119/0.440 | 25s
epoch  5/20 | train 1.870/0.485 | test 2.164/0.445 | 25s
epoch  6/20 | train 1.649/0.538 | test 1.985/0.477 | 25s
epoch  7/20 | train 1.482/0.577 | test 1.675/0.535 | 25s
epoch  8/20 | train 1.321/0.617 | test 1.676/0.559 | 25s
epoch  9/20 | train 1.188/0.653 | test 1.544/0.588 | 26s
epoch 10/20 | train 1.061/0.688 | test 1.515/0.601 | 25s
epoch 11/20 | train 0.944/0.718 | test 1.426/0.622 | 26s
epoch 12/20 | train 0.829/0.753 | test 1.361/0.646 | 26s
epoch 13/20 | train 0.727/0.781 | test 1.310/0.658 | 26s
epoch 14/20 | train 0.626/0.812 | test 1.293/0.668 | 25s
epoch 15/20 | train 0.529/0.844 | test 1.220/0.679 | 25s
epoch 16/20 | train 0.454/0.866 | test 1.192/0.690 | 25s
epoch 17/20 | train 0.388/0.889 | test 1.169/0.693 | 25s
epoch 18/20 | train 0.344/0.907

[('exp_202_wide_conf', [0.7038, 0.7042, 0.704], 20, 3),
 ('exp_208_deep_residual', [0.6908], 20, 1)]

---

## S1c · exp_208 的 3-seed 复核（可选 · ~21 min）

exp_208（深度 ×2 + 残差）是 S1 里第二个正收益，但现在只有 **1 seed**，+1.45 pt 只比噪声底（~1 pt）高一点。
补上 3 seeds，"残差解锁深度"才能从"方向性证据"变成能写进报告的结论。
跑法同 S1b：**新的 run 名 `deep_residual_conf`**（原 1-seed 的 `deep_residual` 保留），`tier=confirm`，主表上两行并存。

不做也不影响 S2 —— 但这是 21 分钟的便宜保险。


code #6 只要 code #1 + code #2 之后就可以直接跑。


In [4]:
# exp_208 的 3-seed 复核（约 21 min，每 seed ~7 min）。只要 code #1 + code #2。
S1C_EPOCHS = 20
S1C_BATCH = [
    {"exp": "exp_208", "name": "deep_residual_conf", "model": cnn_deep_residual, "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep+residual", "confirm")},
]
run_batch(S1C_BATCH, loaders=AUG, epochs=S1C_EPOCHS)


epoch  1/20 | train 3.783/0.125 | test 3.413/0.178 | 21s
epoch  2/20 | train 3.220/0.209 | test 2.990/0.246 | 21s
epoch  3/20 | train 2.705/0.300 | test 2.573/0.332 | 21s
epoch  4/20 | train 2.224/0.403 | test 2.357/0.391 | 21s
epoch  5/20 | train 1.920/0.471 | test 1.971/0.464 | 21s
epoch  6/20 | train 1.705/0.521 | test 2.064/0.456 | 21s
epoch  7/20 | train 1.538/0.562 | test 1.686/0.541 | 21s
epoch  8/20 | train 1.396/0.599 | test 1.622/0.549 | 21s
epoch  9/20 | train 1.260/0.633 | test 1.511/0.590 | 21s
epoch 10/20 | train 1.144/0.663 | test 1.475/0.593 | 20s
epoch 11/20 | train 1.029/0.696 | test 1.479/0.598 | 20s
epoch 12/20 | train 0.928/0.722 | test 1.346/0.635 | 21s
epoch 13/20 | train 0.824/0.749 | test 1.280/0.649 | 21s
epoch 14/20 | train 0.722/0.781 | test 1.264/0.656 | 21s
epoch 15/20 | train 0.635/0.806 | test 1.228/0.672 | 21s
epoch 16/20 | train 0.553/0.834 | test 1.175/0.683 | 21s
epoch 17/20 | train 0.484/0.855 | test 1.160/0.687 | 21s
epoch 18/20 | train 0.434/0.871

[('exp_208_deep_residual_conf', [0.6982, 0.6862, 0.69], 20, 3)]

---

## S2 · 过夜长跑（60 epochs · 3 seeds · 9 个配置一次挂完）★ 报告核心

**顺序 = 优先级**：前四项跑完，"过拟合起点"和"aug 判决"这两个报告硬项就已经拿到手了。
`skip_existing=True` —— 中途掐掉/崩了，重跑本格会**跳过已完成的配置**，从断点继续。

| exp | 配置 | 回答什么问题 | 预估 |
|---|---|---|---|
| **211** | `cnn_wide` × 60ep，**无增强** | 冠军的**过拟合从第几个 epoch 开始**（任务书验收硬项） | 1.3 h |
| **212** | `cnn_wide` × 60ep，**crop+flip** | **aug 在长预算下是否翻案**（任务书点名的"短训练陷阱"） | 1.3 h |
| **213** | `cnn_deep_residual` × 60ep，**无增强** | "残差解锁深度"在长预算下**放大还是消失** | 1.1 h |
| **214** | `cnn_deep_residual` × 60ep，**crop+flip** | 同上，外加这组配置自己的 aug 符号 | 1.1 h |
| **216** | `cnn_deep_residual_wide` × 60ep，增强 | 宽 ×2 与"深 ×2+残差"两个赢面**能否叠加** | 3.2 h |
| **217** | `cnn_deep_residual_d3` × 60ep，增强 | **深度阶梯**第 3 点（3 个残差块/stage，7.9M） | 1.6 h |
| **218** | `cnn_deep_residual_d4` × 60ep，增强 | **深度阶梯**第 4 点（4 个残差块/stage，10.7M） | 2.2 h |
| **219** | `cnn_wide3` × 60ep，增强 | **宽度阶梯**第 3 点（宽 ×3，21.0M） | 2.6 h |
| **220** | `cnn_wide` + **Cutout** × 60ep，增强 | 增强家族第三项：模型开始背题后 Cutout 是否也翻正 | 1.3 h |

**总计 ~15.7 h**（GPU 慢态会让它变成 22 h+；反正掐了就续跑）。

**设计要点**
- **211–214 是 2×2**：aug 的符号在两种架构上各测一次，和第一批的"架构 × 增强"交互（§batch-1）对照。
- **深度阶梯 1→2→3→4 个残差块、宽度阶梯 1×→2×→3×**：两条容量曲线，全部带增强（和 S1 筛选同一正则化区间，阶梯内部互相可比）。
- **216 是"下一步假设"**：宽 ×2（+2.77）和深 ×2+残差（+1.45）两个独立赢面放在一起，看是相加还是互吃。
- **220 相对 212 只多 Cutout 一个变量**（单独的 loader，见 code #2）。
- 本批所有 run 都记 **EMA 曲线**（`ema_decay=0.999`：每 step 更新影子权重、每 epoch 评估一次；只影响记录，不影响训练、不改 model.pt）。
- **epoch 阶梯图不用单独跑**：这 9 条 60ep 曲线的逐 epoch val 就是阶梯。

⚠️ 开跑前：确认笔记本性能模式 + 跑上面那个 15 秒 GPU 自检。⚠️ 跑起来之后**别动这个 notebook**（编辑器保存会覆盖磁盘改动）。


In [3]:
# S2 过夜长跑：9 个配置 x 3 seeds x 60 ep，一次挂上去。只要 code #1 + code #2 就能跑。
# 顺序 = 优先级；断了/掐了就重跑本格，skip_existing 会跳过已完成的配置。
# 每个 run 都会记 EMA 曲线（run_batch 的 ema_decay 默认 0.999，想关掉就在该项写 "ema_decay": None）。
print("9 配置 x 3 seeds x 60ep | 预计 ~15.7 h（慢态 ~22 h+，中断后重跑本格可续）")
S2_EPOCHS = 60

S2_BATCH = [
    # --- 2x2 核心：过拟合起点 + aug 判决 ---
    {"exp": "exp_211", "name": "wide_noaug",     "model": cnn_wide,                "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+wide",              "confirm", aug="none"),      "loaders": NOAUG},
    {"exp": "exp_212", "name": "wide_aug",       "model": cnn_wide,                "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+wide",              "confirm", aug="crop+flip"), "loaders": AUG},
    {"exp": "exp_213", "name": "deepres_noaug",  "model": cnn_deep_residual,       "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep+residual",     "confirm", aug="none"),      "loaders": NOAUG},
    {"exp": "exp_214", "name": "deepres_aug",    "model": cnn_deep_residual,       "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep+residual",     "confirm", aug="crop+flip"), "loaders": AUG},
    # --- 组合：两个赢面能不能叠加 ---
    {"exp": "exp_216", "name": "dr_wide_aug",    "model": cnn_deep_residual_wide,  "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep+residual+wide", "confirm", aug="crop+flip"), "loaders": AUG},
    # --- 深度阶梯（都带残差）---
    {"exp": "exp_217", "name": "dr_d3_aug",      "model": cnn_deep_residual_d3,    "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep3+residual",    "confirm", aug="crop+flip"), "loaders": AUG},
    {"exp": "exp_218", "name": "dr_d4_aug",      "model": cnn_deep_residual_d4,    "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep4+residual",    "confirm", aug="crop+flip"), "loaders": AUG},
    # --- 宽度阶梯 ---
    {"exp": "exp_219", "name": "wide3_aug",      "model": cnn_wide3,               "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+wide3",             "confirm", aug="crop+flip"), "loaders": AUG},
    # --- 增强家族：Cutout（相对 212 只多这一个变量）---
    {"exp": "exp_220", "name": "wide_cutout",    "model": cnn_wide,                "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+wide",              "confirm", aug="crop+flip+erase"), "loaders": ERASE},
]
run_batch(S2_BATCH, epochs=S2_EPOCHS)


9 配置 x 3 seeds x 60ep | 预计 ~15.7 h（慢态 ~22 h+，中断后重跑本格可续）
epoch  1/60 | train 3.632/0.167 | test 3.176/0.242 | ema 4.606/0.010 | 37s
epoch  2/60 | train 2.866/0.278 | test 2.522/0.345 | ema 4.617/0.010 | 27s
epoch  3/60 | train 2.255/0.396 | test 2.369/0.381 | ema 4.643/0.026 | 27s
epoch  4/60 | train 1.763/0.512 | test 2.042/0.457 | ema 4.542/0.027 | 27s
epoch  5/60 | train 1.423/0.590 | test 1.921/0.489 | ema 4.017/0.080 | 27s
epoch  6/60 | train 1.144/0.665 | test 1.782/0.518 | ema 3.217/0.213 | 27s
epoch  7/60 | train 0.860/0.744 | test 1.715/0.547 | ema 2.468/0.369 | 27s
epoch  8/60 | train 0.564/0.830 | test 1.884/0.538 | ema 1.948/0.481 | 27s
epoch  9/60 | train 0.317/0.907 | test 2.035/0.545 | ema 1.649/0.551 | 27s
epoch 10/60 | train 0.160/0.958 | test 1.968/0.567 | ema 1.503/0.589 | 27s
epoch 11/60 | train 0.097/0.976 | test 2.016/0.577 | ema 1.452/0.610 | 27s
epoch 12/60 | train 0.059/0.987 | test 2.208/0.556 | ema 1.460/0.624 | 27s
epoch 13/60 | train 0.114/0.966 | test 2.675

[('exp_211_wide_noaug', [0.617, 0.612, 0.615], 60, 3),
 ('exp_212_wide_aug', [0.722, 0.723, 0.7294], 60, 3),
 ('exp_213_deepres_noaug', [0.5818, 0.5806, 0.5814], 60, 3),
 ('exp_214_deepres_aug', [0.7148, 0.717, 0.712], 60, 3),
 ('exp_216_dr_wide_aug', [0.742, 0.7392, 0.7428], 60, 3),
 ('exp_217_dr_d3_aug', [0.72, 0.721, 0.7184], 60, 3),
 ('exp_218_dr_d4_aug', [0.718, 0.719, 0.7228], 60, 3),
 ('exp_219_wide3_aug', [0.7286, 0.735, 0.7384], 60, 3),
 ('exp_220_wide_cutout', [0.7254, 0.7336, 0.7336], 60, 3)]

---

## S2c · 60 ep 底座锚点（`cnn_down` · ~1.2 h）

两条容量阶梯都缺同一个点：**宽度 ×1 / 深度 1 块 = `cnn_down` @ 60ep**。补上它一次解决三件事：

- 宽度阶梯 1×→2×→3× 完整，深度阶梯 1→2→3→4 完整；
- 60 ep 区块自带**同预算尺子**，每个 Δ 都能直接算；
- 可以严格检验"可加性"：`216 ≈ base + (wide − base) + (dr − base)`。

⚠️ run 名用 `base_down60`，和 20 ep 的 `exp_200_base_down` 是两个不同的预算点，主表里分开看。

code #8 只要 code #1 + code #2 就能跑。


In [3]:
# S2c：60ep 底座锚点（约 1.2 h，3 seeds）。只要 code #1 + code #2。
S2C_EPOCHS = 60
S2C_BATCH = [
    {"exp": "exp_221", "name": "base_down60", "model": cnn_down, "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("downsampled", "anchor", aug="crop+flip"), "loaders": AUG},
]
run_batch(S2C_BATCH, epochs=S2C_EPOCHS)


epoch  1/60 | train 4.002/0.108 | test 3.616/0.157 | ema 4.605/0.011 | 23s
epoch  2/60 | train 3.353/0.190 | test 3.049/0.239 | ema 4.598/0.009 | 9s
epoch  3/60 | train 2.800/0.282 | test 2.681/0.316 | ema 4.523/0.020 | 9s
epoch  4/60 | train 2.340/0.378 | test 2.284/0.393 | ema 4.378/0.029 | 9s
epoch  5/60 | train 2.043/0.444 | test 2.098/0.443 | ema 4.001/0.072 | 9s
epoch  6/60 | train 1.820/0.496 | test 2.012/0.469 | ema 3.402/0.170 | 9s
epoch  7/60 | train 1.649/0.539 | test 1.804/0.509 | ema 2.727/0.306 | 9s
epoch  8/60 | train 1.509/0.572 | test 1.680/0.546 | ema 2.218/0.418 | 9s
epoch  9/60 | train 1.395/0.601 | test 1.645/0.555 | ema 1.855/0.500 | 9s
epoch 10/60 | train 1.294/0.625 | test 1.666/0.550 | ema 1.621/0.551 | 9s
epoch 11/60 | train 1.216/0.647 | test 1.569/0.580 | ema 1.462/0.597 | 9s
epoch 12/60 | train 1.130/0.670 | test 1.627/0.575 | ema 1.350/0.627 | 9s
epoch 13/60 | train 1.056/0.688 | test 1.418/0.615 | ema 1.277/0.647 | 9s
epoch 14/60 | train 0.985/0.706 | tes

[('exp_221_base_down60', [0.692, 0.6916, 0.6984], 60, 3)]

---

## S4 · 评估与集成（不训练 · ~10 min）

三件事，全部复用已经落盘的 `model.pt`：

1. **top-5 回填**：扫 `runs/`，对每个 run 的最终模型在 5k val 上重算 top-1/top-5，顺手核对 top-1 是否等于 `metrics.csv` 末行（不等就说明架构映射写错了）。结果写 `runs/topk_summary.csv`，主表可以直接并进去。
2. **TTA**：原图 + 水平翻转，概率平均 —— 不用任何 test 标签，只是同一输入的多个视角做集成。
3. **3-seed 集成**：同一配置的三个 seed 概率平均（成本 ×3，报告里要说明收益 vs 成本）。

⚠️ 只对 45k/5k 协议的 run 做（`eval_set == "val5k"`）；`runs_final/` 里的 50k 模型不在这里。


In [3]:
# S4a：top-5 回填（只评估不训练，约 5-10 min）。写 runs/topk_summary.csv。
import json
import csv
from src.data import build_loaders_val
from src.engine import evaluate_topk
from src.models import (baseline_cnn, baseline_cnn_bn, cnn_3x3, cnn_down, cnn_residual, cnn_wide,
                        cnn_deep, cnn_3stage, cnn_dw, cnn_dw_wide, cnn_groupnorm, cnn_deep_residual,
                        cnn_deep_residual_wide, cnn_deep_residual_d3, cnn_deep_residual_d4, cnn_wide3)

# config.json 里的 arch 字符串 -> 模型工厂（分批实验写进 config 的描述字段）
ARCH2MODEL = {
    "baseline": baseline_cnn, "baseline+BN": baseline_cnn_bn, "2x3x3 stack": cnn_3x3,
    "downsampled": cnn_down, "down+residual": cnn_residual, "down+wide": cnn_wide,
    "down+deep": cnn_deep, "3stage": cnn_3stage, "down+dw": cnn_dw, "down+dw+wide": cnn_dw_wide,
    "down+gn": cnn_groupnorm, "down+deep+residual": cnn_deep_residual,
    "down+deep+residual+wide": cnn_deep_residual_wide,
    "down+deep3+residual": cnn_deep_residual_d3, "down+deep4+residual": cnn_deep_residual_d4,
    "down+wide3": cnn_wide3,
}
LEGACY = {"exp_000_baseline", "exp_001_adam", "exp_002_adam_45k"}   # 和 03_final_analysis 一致

_, val_loader, _ = build_loaders_val(batch_size=128, num_workers=NUM_WORKERS)
crit = torch.nn.CrossEntropyLoss()

rows, skipped = [], []
for d in sorted(RUNS_DIR.iterdir()):
    if not d.is_dir() or not (d / "model.pt").exists():
        continue
    cfg = json.loads((d / "config.json").read_text(encoding="utf-8"))
    if d.name in LEGACY or cfg.get("eval_set", "val5k") != "val5k":
        skipped.append(d.name)
        continue
    factory = ARCH2MODEL.get(cfg.get("arch"))
    if factory is None:
        skipped.append(d.name)
        print("!! unknown arch:", cfg.get("arch"), d.name)
        continue

    model = factory().to(device)
    model.load_state_dict(torch.load(d / "model.pt", map_location=device, weights_only=True))
    _, t1, t5 = evaluate_topk(model, val_loader, crit, device)

    last = float(list(csv.DictReader(open(d / "metrics.csv", newline="", encoding="utf-8")))[-1]["test_acc"])
    flag = "" if abs(last - t1) < 1e-6 else "   <-- !! differs from metrics.csv (%.4f)" % last

    rows.append({"run": d.name, "exp": cfg.get("exp"), "arch": cfg.get("arch"), "tier": cfg.get("tier"),
                 "aug": cfg.get("aug"), "epochs": cfg.get("epochs"), "seed": cfg.get("seed"),
                 "val_top1": round(t1, 4), "val_top5": round(t5, 4)})
    print("%-36s top1=%.4f top5=%.4f%s" % (d.name, t1, t5, flag))

out = RUNS_DIR / "topk_summary.csv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
print()
print("%d runs -> %s | skipped %d: %s" % (len(rows), out, len(skipped), skipped))


exp_101_baseline_sgd_s0              top1=0.1166 top5=0.3186
exp_101_baseline_sgd_s123            top1=0.1108 top5=0.3222
exp_101_baseline_sgd_s42             top1=0.1126 top5=0.3178
exp_102_adam_s0                      top1=0.2794 top5=0.5774
exp_102_adam_s123                    top1=0.3452 top5=0.6466
exp_102_adam_s42                     top1=0.0076 top5=0.0392
exp_103_bn_s0                        top1=0.4746 top5=0.7756
exp_103_bn_s123                      top1=0.4702 top5=0.7702
exp_103_bn_s42                       top1=0.4704 top5=0.7640
exp_104_lrsched_s0                   top1=0.5420 top5=0.8282
exp_104_lrsched_s123                 top1=0.5398 top5=0.8294
exp_104_lrsched_s42                  top1=0.5410 top5=0.8254
exp_105_aug_s0                       top1=0.4974 top5=0.7994
exp_105_aug_s123                     top1=0.5010 top5=0.7968
exp_105_aug_s42                      top1=0.4994 top5=0.8034
exp_106_arch_3x3_s0                  top1=0.5374 top5=0.8366
exp_106_arch_3x3_s123   

In [3]:
# S4b：冠军 216 的 TTA 与 3-seed 集成（val5k，约 1 min；复用 model.pt，不重训）
from src.data import build_loaders_val
from src.engine import evaluate_topk, ensemble_eval

_, val_loader, _ = build_loaders_val(batch_size=128, num_workers=NUM_WORKERS)
crit = torch.nn.CrossEntropyLoss()

models216 = []
for s in (0, 42, 123):
    m = cnn_deep_residual_wide().to(device)
    m.load_state_dict(torch.load(RUNS_DIR / ("exp_216_dr_wide_aug_s%d" % s) / "model.pt",
                                 map_location=device, weights_only=True))
    models216.append(m)
    _, t1, t5 = evaluate_topk(m, val_loader, crit, device)
    print("seed %-3d              top1=%.4f top5=%.4f" % (s, t1, t5))

for label, ms, tta in [("single seed0  raw", models216[:1], False),
                       ("single seed0  TTA", models216[:1], True),
                       ("3-seed ens    raw", models216,    False),
                       ("3-seed ens    TTA", models216,    True)]:
    t1, t5 = ensemble_eval(ms, val_loader, device, tta=tta)
    print("%-20s top1=%.4f top5=%.4f" % (label, t1, t5))
print()
print("注：这些是 val5k 数字；headline 以 S5 的 test10k 为准（只测一次）。")


seed 0                top1=0.7420 top5=0.9298
seed 42               top1=0.7392 top5=0.9284
seed 123              top1=0.7428 top5=0.9290
single seed0  raw    top1=0.7420 top5=0.9298
single seed0  TTA    top1=0.7484 top5=0.9322
3-seed ens    raw    top1=0.7626 top5=0.9380
3-seed ens    TTA    top1=0.7634 top5=0.9408

注：这些是 val5k 数字；headline 以 S5 的 test10k 为准（只测一次）。


---

## S5 · 最终交付：50k 全量重训 → test 只测一次

- 冠军配方（216：`cnn_deep_residual_wide` + crop+flip + 60 ep），训练集换成 **50k 全量**（5k val 也参与训练）。
- **没有 held-out val**：每 epoch 只在 train 的 2k 子集上记一条监控曲线（不是 test，见 `src/data.py:build_train_test_loaders`）。
- **test 只在训练结束后评估一次**，每 seed 一次 + TTA + 3-seed 集成，全部分开报告、**不做挑选**。
- 产物写 `runs_final/`（和 `runs/` 的 45k/5k 协议分开；主表不会把 test 数字混进去）。
- 成本：50k 比 45k 多 11%，~70 s/epoch → **~72 min/seed × 3 = ~3.6 h**。

⚠️ **看完 test 就不要再改模型了** —— 再改就是拿 test 调参（任务书 3.5 的反面案例）。要改就回到 val 上做。

code #11 依赖 code #1 + code #2（要用 `aug_train_tf` 和 `NUM_WORKERS`）。


In [4]:
# S5：50k 全量重训 + test 只测一次（约 3.6 h）。依赖 code #1 + code #2。
from src.data import build_train_test_loaders
from src.experiments import run_final_retrain

FIN_EPOCHS = 60
fin_train, fin_test, fin_monitor = build_train_test_loaders(
    batch_size=128, num_workers=NUM_WORKERS, train_tf=aug_train_tf, monitor_size=2000)

final_summary = run_final_retrain(
    "exp_230", "final_dr_wide", cnn_deep_residual_wide,
    train_loader=fin_train, test_loader=fin_test, monitor_loader=fin_monitor,
    extra=build_experiment_metadata("down+deep+residual+wide", "final", aug="crop+flip"),
    seeds=(0, 42, 123), epochs=FIN_EPOCHS, batch_size=128, device=device, tta=True)


epoch  1/60 | train 3.542/0.160 | test 2.980/0.268 | 80s
epoch  2/60 | train 3.000/0.247 | test 2.651/0.327 | 66s
epoch  3/60 | train 2.469/0.352 | test 2.509/0.359 | 66s
epoch  4/60 | train 1.990/0.455 | test 2.132/0.423 | 66s
epoch  5/60 | train 1.716/0.517 | test 1.635/0.551 | 66s
epoch  6/60 | train 1.528/0.562 | test 1.412/0.595 | 66s
epoch  7/60 | train 1.370/0.605 | test 1.418/0.593 | 66s
epoch  8/60 | train 1.244/0.636 | test 1.570/0.578 | 66s
epoch  9/60 | train 1.133/0.665 | test 1.387/0.599 | 67s
epoch 10/60 | train 1.035/0.690 | test 0.948/0.711 | 66s
epoch 11/60 | train 0.933/0.718 | test 0.956/0.700 | 66s
epoch 12/60 | train 0.857/0.738 | test 0.831/0.750 | 66s
epoch 13/60 | train 0.780/0.760 | test 0.748/0.759 | 66s
epoch 14/60 | train 0.706/0.780 | test 0.710/0.774 | 66s
epoch 15/60 | train 0.639/0.799 | test 0.600/0.803 | 66s
epoch 16/60 | train 0.568/0.820 | test 0.479/0.850 | 66s
epoch 17/60 | train 0.503/0.838 | test 0.492/0.847 | 67s
epoch 18/60 | train 0.440/0.858